# 06 — Paradox Analysis


In [1]:
import sys
import json
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
from src.data.loaders import load_text_data, load_structured_data
from src.data.preprocessing import compute_paradox_statistics
from src.utils import ensure_dir

sns.set_theme(style="whitegrid")


In [2]:
text_df = load_text_data(sample=10000)
struct_df = load_structured_data()
text_stats = compute_paradox_statistics(text_df)
struct_df["aosp_proxy"] = (
    struct_df["Social_Comparison_Trigger"] * 0.4
    + (struct_df["Daily_Screen_Time_Hours"] / struct_df["Daily_Screen_Time_Hours"].max()) * 0.3
    + struct_df["Late_Night_Usage"] * 0.3
)
paradox = {
    **text_stats,
    "structured_aosp_distress_correlation": float(struct_df["aosp_proxy"].corr(struct_df["GAD_7_Score"] + struct_df["PHQ_9_Score"])),
    "structured_paradox_rate": float(((struct_df["aosp_proxy"] >= struct_df["aosp_proxy"].median()) & (struct_df["high_distress"] == 1)).mean()),
}
for k, v in paradox.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


aosp_distress_correlation: -0.0451
paradox_index_correlation: 0.1973
high_aosp_high_distress_rate: 0.6876
mean_aosp: 0.0009
mean_distress: 0.6876
mean_paradox_index: 0.0000
structured_aosp_distress_correlation: 0.7517
structured_paradox_rate: 0.3586


In [3]:
out = ensure_dir(ROOT / "outputs" / "results")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sample = text_df.sample(min(3000, len(text_df)), random_state=42)
sns.scatterplot(data=sample, x="aosp_composite", y="distress_score", hue="wellbeing_decline",
                alpha=0.35, palette=["#4C72B0", "#DD8452"], ax=axes[0])
axes[0].set_title("AOSP vs Distress")
sns.scatterplot(data=struct_df, x="aosp_proxy", y="GAD_7_Score", hue="high_distress",
                alpha=0.35, palette=["#4C72B0", "#DD8452"], ax=axes[1])
axes[1].set_title("Behavioural AOSP vs GAD-7")
plt.tight_layout()
plt.savefig(out / "paradox_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
with open(out / "paradox_statistics.json", "w") as f:
    json.dump(paradox, f, indent=2)
print("06_paradox_analysis.py complete.")


06_paradox_analysis.py complete.
